In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

STAGE1_DIR = Path(
    r"C:\Users\15195\Desktop\coding part\dataset merged"
)

OUTPUT_DIR = Path(
    r"C:\Users\15195\Desktop"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\15195\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Read the transaction-product merged file
transaction_product_path = STAGE1_DIR / "transaction_product_merged.csv"

# Read household demographic lookup file separately
demographic_path = STAGE1_DIR / "transaction_demographic_merged.csv"
df = pd.read_csv(transaction_product_path)
hh_demographic = pd.read_csv(demographic_path)

print("Transaction-product dataset shape:", df.shape)
print("Demographic lookup shape:", hh_demographic.shape)

print("\nTransaction-product columns:")
print(df.columns.tolist())

print("\nDemographic columns:")
print(hh_demographic.columns.tolist())

Transaction-product dataset shape: (2595732, 18)
Demographic lookup shape: (2595732, 20)

Transaction-product columns:
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']

Demographic columns:
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'AGE_DESC', 'MARITAL_STATUS_CODE', 'INCOME_DESC', 'HOMEOWNER_DESC', 'HH_COMP_DESC', 'HOUSEHOLD_SIZE_DESC', 'KID_CATEGORY_DESC', '_merge']


In [3]:
# Calculate the latest observed transaction day
max_day = df["DAY"].max()

# Create household-level behavioural features
household_features = df.groupby("household_key").agg(
    recency=("DAY", lambda x: max_day - x.max()),
    frequency=("BASKET_ID", "nunique"),
    monetary=("SALES_VALUE", "sum"),
    total_quantity=("QUANTITY", "sum"),
    unique_products=("PRODUCT_ID", "nunique"),
    category_diversity=("COMMODITY_DESC", "nunique"),
    first_purchase_day=("DAY", "min"),
    last_purchase_day=("DAY", "max")
).reset_index()

# Calculate average basket value
household_features["average_basket_value"] = (
    household_features["monetary"]
    / household_features["frequency"]
)

# Calculate average number of unique products per basket
basket_level = (
    df.groupby(["household_key", "BASKET_ID"])
      .agg(
          basket_unique_products=("PRODUCT_ID", "nunique")
      )
      .reset_index()
)

average_basket_size = (
    basket_level.groupby("household_key")
    .agg(
        average_basket_size=("basket_unique_products", "mean")
    )
    .reset_index()
)

household_features = household_features.merge(
    average_basket_size,
    on="household_key",
    how="left",
    validate="one_to_one"
)

# Calculate observed customer lifetime
household_features["customer_lifetime_days"] = (
    household_features["last_purchase_day"]
    - household_features["first_purchase_day"]
)

print("Household feature dataset shape:", household_features.shape)
print("Number of households:", household_features["household_key"].nunique())

household_features.head()

Household feature dataset shape: (2500, 12)
Number of households: 2500


,household_key,recency,frequency,monetary,total_quantity,unique_products,category_diversity,first_purchase_day,last_purchase_day,average_basket_value,average_basket_size,customer_lifetime_days
0,1,5,86,4330.16,1997,677,128,51,706,50.350698,20.081395,655
1,2,43,45,1954.34,834,546,141,103,668,43.429778,15.866667,565
2,3,8,47,2653.21,8540,516,110,113,703,56.451277,19.617021,590
3,4,84,30,1200.11,382,164,63,104,627,40.003667,10.033333,523
4,5,8,40,779.06,245,199,76,85,703,19.476500,5.550000,618


In [4]:
# Calculate household-level promotion and discount features
discount_features = df.groupby("household_key").agg(
    total_retail_discount=("RETAIL_DISC", "sum"),
    total_coupon_discount=("COUPON_DISC", "sum"),
    total_coupon_match_discount=("COUPON_MATCH_DISC", "sum")
).reset_index()

household_features = household_features.merge(
    discount_features,
    on="household_key",
    how="left",
    validate="one_to_one"
)

# Calculate overall discount amount
household_features["total_discount"] = (
    household_features["total_retail_discount"].abs()
    + household_features["total_coupon_discount"].abs()
    + household_features["total_coupon_match_discount"].abs()
)

# Calculate discount sensitivity relative to total sales value
household_features["discount_ratio"] = (
    household_features["total_discount"]
    / household_features["monetary"]
)

# Optional: retain coupon-specific discount ratio
household_features["coupon_discount_ratio"] = (
    (
        household_features["total_coupon_discount"].abs()
        + household_features["total_coupon_match_discount"].abs()
    )
    / household_features["monetary"]
)

print(
    household_features[
        [
            "total_discount",
            "discount_ratio",
            "coupon_discount_ratio"
        ]
    ].describe()
)

       total_discount  discount_ratio  coupon_discount_ratio
count     2500.000000     2500.000000            2500.000000
mean       579.408876        0.190654               0.005322
std        593.601633        0.082162               0.010521
min          0.250000        0.009884               0.000000
25%        170.407500        0.136334               0.000000
50%        396.275000        0.177572               0.001859
75%        779.332500        0.228058               0.005745
max       5162.450000        0.785938               0.137990


In [5]:
# Select behavioural variables for K-means clustering
clustering_variables = [
    "recency",
    "frequency",
    "monetary",
    "total_quantity",
    "unique_products",
    "category_diversity",
    "average_basket_value",
    "average_basket_size",
    "customer_lifetime_days",
    "discount_ratio"
]

# Check for missing and infinite values before clustering
print("Missing values in clustering variables:")
print(household_features[clustering_variables].isnull().sum())

print("\nInfinite values in clustering variables:")
print(
    np.isinf(
        household_features[clustering_variables]
    ).sum()
)



Missing values in clustering variables:
recency                   0
frequency                 0
monetary                  0
total_quantity            0
unique_products           0
category_diversity        0
average_basket_value      0
average_basket_size       0
customer_lifetime_days    0
discount_ratio            0
dtype: int64

Infinite values in clustering variables:
recency                   0
frequency                 0
monetary                  0
total_quantity            0
unique_products           0
category_diversity        0
average_basket_value      0
average_basket_size       0
customer_lifetime_days    0
discount_ratio            0
dtype: int64


In [6]:
# Save household behavioural features for clustering
household_features.to_csv(
    OUTPUT_DIR / "household_features_for_clustering.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "household_features_for_clustering.csv")

Saved: C:\Users\15195\Desktop\household_features_for_clustering.csv


In [7]:
# Check whether household_features has one row per household
print("Rows in household_features:", household_features.shape[0])
print("Unique households:", household_features["household_key"].nunique())
print(
    "Duplicate household_key:",
    household_features["household_key"].duplicated().sum()
)

# Display duplicate households if they exist
duplicate_households = household_features[
    household_features["household_key"].duplicated(keep=False)
].sort_values("household_key")

print(duplicate_households.head(20))

Rows in household_features: 2500
Unique households: 2500
Duplicate household_key: 0
Empty DataFrame
Columns: [household_key, recency, frequency, monetary, total_quantity, unique_products, category_diversity, first_purchase_day, last_purchase_day, average_basket_value, average_basket_size, customer_lifetime_days, total_retail_discount, total_coupon_discount, total_coupon_match_discount, total_discount, discount_ratio, coupon_discount_ratio]
Index: []


In [8]:
# Read the original household-level demographic lookup table again
hh_demographic_lookup = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\old dataset\hh_demographic.csv"
)

# Check the demographic table structure
print("Demographic lookup shape:", hh_demographic_lookup.shape)

print(
    "Duplicate household_key in demographic lookup:",
    hh_demographic_lookup["household_key"].duplicated().sum()
)

print(
    "Unique households in demographic lookup:",
    hh_demographic_lookup["household_key"].nunique()
)

# Merge demographic variables only for later segment profiling
household_features_profile = household_features.merge(
    hh_demographic_lookup,
    on="household_key",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nDemographic merge result:")
print(household_features_profile["_merge"].value_counts())

print(
    "\nHouseholds with available demographic information:",
    household_features_profile["AGE_DESC"].notna().sum()
)

# Remove merge-status column before saving
household_features_profile = household_features_profile.drop(
    columns="_merge"
)

# Save profiling dataset
household_features_profile.to_csv(
    OUTPUT_DIR / "household_features_with_demographics.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "household_features_with_demographics.csv"
)

Demographic lookup shape: (801, 8)
Duplicate household_key in demographic lookup: 0
Unique households in demographic lookup: 801

Demographic merge result:
_merge
left_only     1699
both           801
right_only       0
Name: count, dtype: int64

Households with available demographic information: 801
Saved: C:\Users\15195\Desktop\household_features_with_demographics.csv
